# Score Analysis for Retrievers

In [ ]:
import sys
sys.path.insert(0, '/Users/skyler/Projects/document_retrieval_project')

from loader import load_data
from retrievers.bm25 import BM25Retriever
from retrievers.tf_idf_retriever import TFIDFRetriever
from retrievers.dense_retriever import DenseRetriever
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load dataset and build corpus
ds = load_data()
passages_text = []
for example in ds:
    for passage in example["passages"]["passage_text"]:
        passages_text.append(passage)

print(f"Total passages: {len(passages_text)}")

In [ ]:
# Fit retrievers
bm25 = BM25Retriever(top_k=10)
tfidf = TFIDFRetriever(top_k=10)
dense = DenseRetriever(top_k=10)

bm25.fit(passages_text)
tfidf.fit(passages_text)
dense.fit("sbert_embeddings.npy", passages_text=passages_text)
print("All retrievers fitted")

In [ ]:
# Check scores for a single query
query = "What is the capital of France?"

bm25_scores = np.array(bm25.score(query))
tfidf_scores = np.array(tfidf.score(query))
dense_scores = np.array(dense.score(query))

print(f"BM25  — min: {bm25_scores.min():.4f}, max: {bm25_scores.max():.4f}, nonzero: {np.count_nonzero(bm25_scores)}")
print(f"TF-IDF — min: {tfidf_scores.min():.4f}, max: {tfidf_scores.max():.4f}, nonzero: {np.count_nonzero(tfidf_scores)}")
print(f"Dense  — min: {dense_scores.min():.4f}, max: {dense_scores.max():.4f}, nonzero: {np.count_nonzero(dense_scores)}")

In [ ]:
# Check top-10 passages for each retriever
bm25_ids = bm25.query(query)
tfidf_ids = tfidf.query(query)
dense_ids = dense.query(query)

print("BM25 top-10 passages:")
for i, idx in enumerate(bm25_ids):
    print(f"  {i+1}. [id={idx}] {passages_text[idx][:100]}...")

print("\nTF-IDF top-10 passages:")
for i, idx in enumerate(tfidf_ids):
    print(f"  {i+1}. [id={idx}] {passages_text[idx][:100]}...")

print("\nDense top-10 passages:")
for i, idx in enumerate(dense_ids):
    print(f"  {i+1}. [id={idx}] {passages_text[idx][:100]}...")

In [ ]:
# Score distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(bm25_scores, bins=50)
axes[0].set_title("BM25 Score Distribution")
axes[0].set_xlabel("Score")

axes[1].hist(tfidf_scores, bins=50)
axes[1].set_title("TF-IDF Score Distribution")
axes[1].set_xlabel("Score")

axes[2].hist(dense_scores, bins=50)
axes[2].set_title("Dense Score Distribution")
axes[2].set_xlabel("Score")

plt.tight_layout()
plt.show()

In [ ]:
# Check union size and hybrid score for the same query
union_ids = set(bm25_ids) | set(tfidf_ids) | set(dense_ids)
print(f"Union size: {len(union_ids)}")

sparse_scores = 0.5 * bm25_scores + 0.5 * tfidf_scores
for idx in union_ids:
    hybrid = 0.5 * sparse_scores[idx] + 0.5 * dense_scores[idx]
    print(f"  id={idx} bm25={bm25_scores[idx]:.4f} tfidf={tfidf_scores[idx]:.4f} dense={dense_scores[idx]:.4f} hybrid={hybrid:.4f}")